# **Alur Kerja Pra-Pemodelan dan Prapemrosesan Data**
Catatan ini menguraikan tahapan esensial dalam prapemrosesan data sebelum memasuki fase pelatihan model machine learning. Fokus utama pembahasan mencakup strategi penanganan nilai yang hilang (missing values), penskalaan fitur (feature scaling), penyandian data kategorikal (categorical encoding), pemanfaatan arsitektur pipeline, hingga prinsip dasar rekayasa fitur (feature engineering).

Tujuan Pembelajaran
Setelah mempelajari modul ini, pembaca diharapkan mampu:

Memahami dampak kualitas data mentah terhadap performa algoritma prediktif.

Mengimplementasikan berbagai teknik imputasi untuk menangani missing data.

Menerapkan metode penskalaan fitur yang sesuai dengan karakteristik algoritma sasaran.

Mengonversi variabel kategorikal menjadi representasi matematis menggunakan teknik encoding.

Membangun alur kerja prapemrosesan hibrida menggunakan ColumnTransformer dan Pipeline.

Mengidentifikasi dan mencegah fenomena kebocoran data (data leakage) selama fase pemodelan.

## Bagian 1: Urgensi Kualitas Data dan Penanganan Data Hilang (Missing Data)
Data mentah di dunia nyata lazimnya dipenuhi oleh nilai yang hilang, anomali (outliers), perbedaan magnitudo skala, hingga variasi tipe data. Mengabaikan masalah ini akan mendistorsi pola yang dipelajari model dan merusak akurasi prediksi.

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Simulasi Dataset dengan Missing Values (NaN)
np.random.seed(42)
df = pd.DataFrame({
    "A": [10, 20, np.nan, 40, 50],
    "B": [5, np.nan, 15, 20, 25],
    "C": [100, 120, 130, np.nan, 180]
})

# 1. Pendekatan SimpleImputer (Mean)
imputer_simple = SimpleImputer(strategy="mean")
df_simple = pd.DataFrame(imputer_simple.fit_transform(df), columns=df.columns)

# 2. Pendekatan KNNImputer
imputer_knn = KNNImputer(n_neighbors=2)
df_knn = pd.DataFrame(imputer_knn.fit_transform(df), columns=df.columns)

# 3. Pendekatan IterativeImputer
imputer_iter = IterativeImputer(random_state=42)
df_iter = pd.DataFrame(imputer_iter.fit_transform(df), columns=df.columns)

display(df_iter)

,A,B,C
0,10.00000,5.000000,100.000000
1,20.00000,10.853627,120.000000
2,25.00043,15.000000,130.000000
3,40.00000,20.000000,160.000047
4,50.00000,25.000000,180.000000


## **Bagian 2: Penskalaan Fitur (Feature Scaling)**
Penskalaan fitur adalah tahapan mutlak bagi algoritma berbasis kalkulasi metrik jarak (seperti KNN dan SVM) atau metode optimasi turunan (seperti Jaringan Syaraf Tiruan dan Regresi Logistik).

In [2]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer

# 1. Implementasi StandardScaler
scaler_std = StandardScaler()
scaled_std = scaler_std.fit_transform(df_iter)

# 2. Implementasi MinMaxScaler
scaler_mm = MinMaxScaler()
scaled_mm = scaler_mm.fit_transform(df_iter)

# 3. Implementasi Normalizer
scaler_norm = Normalizer()
scaled_norm = scaler_norm.fit_transform(df_iter)

# Visualisasi hasil Normalizer
display(pd.DataFrame(scaled_norm, columns=df.columns))

,A,B,C
0,0.099381,0.049690,0.993808
1,0.163749,0.088863,0.982492
2,0.187650,0.112588,0.975762
3,0.240772,0.120386,0.963087
4,0.265279,0.132640,0.955005


## **Bagian 3: Penyandian Variabel Kategorikal (Categorical Encoding)**

Model analitik komputasional membutuhkan masukan matriks numerik. Variabel diskrit yang berisi teks (kategori) wajib dikonversi.OneHotEncoder: Mengonversi setiap kategori unik menjadi kolom biner (dummy variables). Sangat direkomendasikan untuk kategori nominal (tanpa urutan).LabelEncoder: Memetakan kategori menjadi representasi bilangan bulat linear. Hanya disarankan untuk variabel target ($y$), bukan matriks fitur ($X$).Jika sebuah dataset memiliki campuran fitur numerik dan kategorikal, eksekusi pemrosesan paralel dapat dikelola menggunakan ColumnTransformer.

In [3]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer

# Simulasi Dataset Campuran (Mixed Data Types)
mixed_df = pd.DataFrame({
    "Age": [25, 30, 35, 40],
    "Salary": [5000, 7000, 8000, 10000],
    "Department": ["IT", "HR", "Finance", "IT"]
})

# Implementasi Pemrosesan Hibrida menggunakan ColumnTransformer
transformer = ColumnTransformer([
    ("num_scaler", StandardScaler(), ["Age", "Salary"]),
    ("cat_encoder", OneHotEncoder(), ["Department"])
])

result_matrix = transformer.fit_transform(mixed_df)
print("Hasil Transformasi Hibrida:\n", result_matrix)

Hasil Transformasi Hibrida:
 [[-1.34164079 -1.38675049  0.          0.          1.        ]
 [-0.4472136  -0.2773501   0.          1.          0.        ]
 [ 0.4472136   0.2773501   1.          0.          0.        ]
 [ 1.34164079  1.38675049  0.          0.          1.        ]]


## **Bagian 4: Arsitektur Pipeline dan Mitigasi Kebocoran Data**
Kebocoran data (data leakage)

terjadi apabila informasi dari himpunan uji (test set) secara tidak sengaja memengaruhi fase pelatihan (training set). Kesalahan fundamental yang paling sering dilakukan adalah mengaplikasikan fungsi .fit_transform() pada keseluruhan dataset sebelum melakukan partisi.

Praktik Metodologi yang Benar:

Pisahkan dataset (Train/Test Split) terlebih dahulu.

Latih parameter prapemrosesan (metode .fit()) hanya pada himpunan latih.

Terapkan perhitungan tersebut (metode .transform()) pada himpunan uji.

Untuk mengeliminasi risiko kesalahan komputasi manual ini, pustaka menyertakan kelas Pipeline yang mengikat tahapan transformer dan estimator ke dalam satu objek utuh.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Persiapan Data
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, random_state=42)

# Konstruksi Arsitektur Pipeline
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

# Eksekusi Terintegrasi
pipe.fit(X_train, y_train)
print(f"Akurasi Pipeline Terintegrasi: {pipe.score(X_test, y_test):.4f}")

Akurasi Pipeline Terintegrasi: 0.9790


## **Bagian 5: Rekayasa Fitur (Feature Engineering)**
Rekayasa fitur adalah seni mentransformasi atau mengekstraksi atribut baru dari raw data untuk menonjolkan pola laten yang krusial bagi model. Berbeda dengan prapemrosesan mekanis, rekayasa fitur sangat menuntut pemahaman terhadap domain masalah (domain knowledge).

Contoh implementasi rekayasa fitur:

Menghitung rasio penjualan terhadap ketersediaan stok.

Mengonversi variabel tanggal lahir menjadi representasi interval (kategori umur).

Mengagregasi data deret waktu menjadi akumulasi total transaksi bulanan.